# ETHICS deontology repeated-sampling notebook

This notebook:

- loads a JSONL file of prompts
- runs each example 10 times across a temperature schedule
- computes per-question accuracy
- isolates low-accuracy questions
- exports CSVs with only the `scenario` and `excuse` fields plus results

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

DATA_PATH = Path("ethics_deontology_prompts.jsonl")
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

N = None
SEED = 42
BATCH_SIZE = 8
MAX_NEW_TOKENS = 6
N_RUNS = 10
TEMPERATURES = [0.0, 0.1, 0.2, 0.35, 0.5, 0.65, 0.8, 0.95, 1.1, 1.25]
LOW_ACCURACY_THRESHOLD = 0.50

assert len(TEMPERATURES) == N_RUNS, "TEMPERATURES must match N_RUNS"

In [ ]:
def load_jsonl(path: Path, n=None):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    if n is not None:
        rows = rows[:n]
    return rows

data = load_jsonl(DATA_PATH, N)
print("Loaded:", len(data))
print("Keys:", list(data[0].keys()))
print("Sample target:", data[0].get("target_text"), data[0].get("target_label"))
print(data[0]["prompt"][:500])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.pad_token_id
model.eval()

print("Loaded model:", MODEL_ID)
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)

In [ ]:
SCENARIO_PATTERNS = [
    re.compile(r"Scenario:\s*(.*?)(?:\n\s*Excuse:|\n\s*Question:|\n\s*Task:|\n\s*Answer:|$)", re.IGNORECASE | re.DOTALL),
    re.compile(r"Action:\s*(.*?)(?:\n\s*Excuse:|\n\s*Question:|\n\s*Task:|\n\s*Answer:|$)", re.IGNORECASE | re.DOTALL),
]
EXCUSE_PATTERNS = [
    re.compile(r"Excuse:\s*(.*?)(?:\n\s*Question:|\n\s*Task:|\n\s*Answer:|$)", re.IGNORECASE | re.DOTALL),
    re.compile(r"Reason:\s*(.*?)(?:\n\s*Question:|\n\s*Task:|\n\s*Answer:|$)", re.IGNORECASE | re.DOTALL),
]

def _first_match(text, patterns):
    for pat in patterns:
        m = pat.search(text)
        if m:
            return " ".join(m.group(1).strip().split())
    return ""

def extract_scenario_excuse(prompt: str):
    scenario = _first_match(prompt, SCENARIO_PATTERNS)
    excuse = _first_match(prompt, EXCUSE_PATTERNS)

    if not scenario:
        lines = [ln.strip() for ln in prompt.splitlines() if ln.strip()]
        non_meta = []
        for ln in lines:
            low = ln.lower()
            if low.startswith(("question:", "task:", "answer:", "label:", "output:")):
                continue
            non_meta.append(ln)
        if non_meta:
            scenario = non_meta[0]
            if len(non_meta) > 1:
                excuse = non_meta[1]

    return scenario, excuse

for ex in data:
    scenario, excuse = extract_scenario_excuse(ex["prompt"])
    ex["scenario"] = scenario
    ex["excuse"] = excuse

preview = pd.DataFrame(
    [{"scenario": x["scenario"], "excuse": x["excuse"], "target": x.get("target_text")} for x in data[:5]]
)
preview

In [ ]:
YES_STR = " Yes"
NO_STR = " No"

yes_ids = tokenizer.encode(YES_STR, add_special_tokens=False)
no_ids = tokenizer.encode(NO_STR, add_special_tokens=False)

def ensure_answer_slot(prompt: str) -> str:
    p = prompt.rstrip()
    low = p.lower()
    if low.endswith("answer:"):
        return p + " "
    if low.endswith("answer: yes or no"):
        return p + " "
    return p + "\nAnswer: "

def parse_verdict(text: str):
    t = text.strip().lower()
    m = re.search(r"\b(yes|no)\b", t)
    if m:
        return 1 if m.group(1) == "yes" else 0
    return None

def fallback_yes_no_from_logits(batch_prompts):
    prompts2 = [ensure_answer_slot(p) for p in batch_prompts]
    toks = tokenizer(prompts2, return_tensors="pt", padding=True, truncation=True).to(model.device)
    with torch.no_grad():
        logits = model(**toks).logits
    last_idx = toks["attention_mask"].sum(dim=1) - 1
    next_logits = logits[torch.arange(logits.size(0), device=model.device), last_idx]

    if len(yes_ids) == 1 and len(no_ids) == 1:
        ly = next_logits[:, yes_ids[0]]
        ln = next_logits[:, no_ids[0]]
        return (ly > ln).long().cpu().tolist()

    preds = []
    for i in range(next_logits.size(0)):
        ly = next_logits[i, yes_ids[0]]
        ln = next_logits[i, no_ids[0]]
        preds.append(int(ly > ln))
    return preds

def generate_batch_verdicts(batch_prompts, temperature):
    prompts2 = [ensure_answer_slot(p) for p in batch_prompts]
    toks = tokenizer(prompts2, return_tensors="pt", padding=True, truncation=True).to(model.device)

    gen_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if temperature <= 0:
        gen_kwargs["do_sample"] = False
    else:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = 0.95

    with torch.no_grad():
        out = model.generate(**toks, **gen_kwargs)

    input_len = toks["input_ids"].shape[1]
    decoded = tokenizer.batch_decode(out[:, input_len:], skip_special_tokens=True)

    preds = []
    parsed_ok = []
    for txt in decoded:
        p = parse_verdict(txt)
        preds.append(p)
        parsed_ok.append(p is not None)

    if not all(parsed_ok):
        fallback = fallback_yes_no_from_logits(batch_prompts)
        preds = [fb if p is None else p for p, fb in zip(preds, fallback)]

    return preds, decoded, parsed_ok

In [ ]:
ys = [int(ex["target_label"]) for ex in data]
prompts = [ex["prompt"] for ex in data]

all_trial_cols = {}
all_text_cols = {}
all_parsed_cols = {}
temp_summary = []

for run_idx, temp in enumerate(TEMPERATURES):
    run_preds = []
    run_text = []
    run_parsed = []

    for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc=f"temp={temp}"):
        batch_prompts = prompts[i:i+BATCH_SIZE]
        preds, texts, parsed = generate_batch_verdicts(batch_prompts, temp)
        run_preds.extend(preds)
        run_text.extend(texts)
        run_parsed.extend(parsed)

    all_trial_cols[f"trial_{run_idx+1}_temp_{temp}"] = run_preds
    all_text_cols[f"trial_{run_idx+1}_text"] = run_text
    all_parsed_cols[f"trial_{run_idx+1}_parsed_directly"] = run_parsed

    run_acc = sum(int(p == y) for p, y in zip(run_preds, ys)) / len(ys)
    temp_summary.append({
        "run": run_idx + 1,
        "temperature": temp,
        "accuracy": run_acc,
        "pred_rate_yes": sum(run_preds) / len(run_preds),
        "direct_parse_rate": sum(run_parsed) / len(run_parsed),
    })
    print(f"Run {run_idx+1}/{N_RUNS} | temp={temp} | acc={run_acc:.4f}")

temp_df = pd.DataFrame(temp_summary)
temp_df

In [ ]:
result_rows = []

for idx, ex in enumerate(data):
    trial_preds = [all_trial_cols[col][idx] for col in all_trial_cols]
    trial_texts = [all_text_cols[col][idx] for col in all_text_cols]
    trial_parsed = [all_parsed_cols[col][idx] for col in all_parsed_cols]

    y = int(ex["target_label"])
    correct_count = sum(int(p == y) for p in trial_preds)
    acc_rate = correct_count / N_RUNS
    majority_pred = 1 if sum(trial_preds) >= (N_RUNS / 2) else 0

    row = {
        "row_index": idx,
        "group_id": int(ex.get("group_id", -1)),
        "target_label": y,
        "target_text": ex.get("target_text", "Yes" if y == 1 else "No"),
        "scenario": ex.get("scenario", ""),
        "excuse": ex.get("excuse", ""),
        "correct_count": correct_count,
        "accuracy_rate": acc_rate,
        "majority_pred_label": majority_pred,
        "majority_pred_text": "Yes" if majority_pred == 1 else "No",
        "majority_correct": int(majority_pred == y),
        "mean_pred_yes": sum(trial_preds) / N_RUNS,
        "direct_parse_rate": sum(trial_parsed) / N_RUNS,
    }

    for col, vals in all_trial_cols.items():
        row[col] = vals[idx]
    for col, vals in all_text_cols.items():
        row[col] = vals[idx]
    for col, vals in all_parsed_cols.items():
        row[col] = vals[idx]

    result_rows.append(row)

results_df = pd.DataFrame(result_rows).sort_values(["accuracy_rate", "group_id", "row_index"]).reset_index(drop=True)

overall_majority_acc = results_df["majority_correct"].mean()
overall_trial_acc = temp_df["accuracy"].mean()

print(f"Mean per-trial accuracy across temperatures: {overall_trial_acc:.4f}")
print(f"Majority-vote accuracy: {overall_majority_acc:.4f}")
results_df.head()

In [ ]:
low_acc_df = results_df[results_df["accuracy_rate"] < LOW_ACCURACY_THRESHOLD].copy()
print("Low-accuracy count:", len(low_acc_df))
low_acc_df[["row_index", "group_id", "accuracy_rate", "target_text", "majority_pred_text", "scenario", "excuse"]].head(20)

In [ ]:
compact_cols = [
    "row_index",
    "group_id",
    "target_label",
    "target_text",
    "scenario",
    "excuse",
    "correct_count",
    "accuracy_rate",
    "majority_pred_label",
    "majority_pred_text",
    "majority_correct",
    "mean_pred_yes",
    "direct_parse_rate",
]
trial_pred_cols = [c for c in results_df.columns if c.startswith("trial_") and "_temp_" in c]
compact_export = results_df[compact_cols + trial_pred_cols].copy()

compact_export.to_csv("ethics_deontology_per_question_accuracy.csv", index=False)
low_acc_df[compact_cols + trial_pred_cols].to_csv("ethics_deontology_low_accuracy.csv", index=False)
temp_df.to_csv("ethics_deontology_temperature_summary.csv", index=False)

print("Wrote ethics_deontology_per_question_accuracy.csv")
print("Wrote ethics_deontology_low_accuracy.csv")
print("Wrote ethics_deontology_temperature_summary.csv")

## Notes

- `trial_k_temp_t` columns store the predicted label for each run (`1 = Yes`, `0 = No`).
- `accuracy_rate` is per-question accuracy across all 10 runs.
- `majority_pred_*` summarizes the vote across the 10 runs.
- `scenario` and `excuse` are extracted from the original prompt; if the prompt format changes, adjust the regex patterns in the extraction cell.
- For faster dry runs, set `N` to a smaller number before running the notebook.